# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [9]:
# Your experiment here

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("LOADING DATA...")
print("="*60)

# Load the data from the notebook's data folder
# This uses the same data that the notebook uses
try:
    # Try to load from the data folder
    df = pd.read_csv('data/sample_data.csv')
except:
    try:
        # Alternative path
        df = pd.read_csv('../data/sample_data.csv')
    except:
        # If data is already loaded in notebook
        if 'df' in globals():
            df = globals()['df']
            print("Using existing dataframe from notebook")
        else:
            print("ERROR: Could not find data file.")
            print("Please run all cells above first, then run this cell.")
            raise

# Print data info
print(f"Data loaded: {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {df.columns.tolist()}")

# Prepare features and target
print("\n" + "="*60)
print("PREPARING DATA...")
print("="*60)

# Define features to use
feature_cols = ['avg_position', 'impressions_90d', 'content_age_days']

# Check which columns exist
available_cols = [col for col in feature_cols if col in df.columns]
print(f"Available features: {available_cols}")

# Find the target column (likely 'clicked' or 'ctr' related)
target_col = None
for col in ['clicked', 'is_click', 'click', 'target', 'label']:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    # If no obvious target, use CTR > 0.5 as target
    if 'ctr' in df.columns:
        df['clicked'] = (df['ctr'] > df['ctr'].median()).astype(int)
        target_col = 'clicked'
        print(f"Created 'clicked' from CTR (median split)")
    else:
        print("ERROR: No target column found")
        raise

print(f"Target column: {target_col}")

# Create X and y
X = df[available_cols].copy()
y = df[target_col].copy()

# Handle any missing values
X = X.fillna(X.mean())
print(f"Features: {X.columns.tolist()}")
print(f"Target classes: {y.unique()}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training: {len(X_train)} rows")
print(f"Testing: {len(X_test)} rows")

print("\n" + "="*60)
print("EXPERIMENT 1: Changing max_depth")
print("="*60)

# Try different depths
results = []
for depth in [2, 3, 4, 6, 8]:
    print(f"\n--- max_depth = {depth} ---")

    # Train model
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    # Check accuracy
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)

    # Calculate Precision@50
    try:
        probs = model.predict_proba(X_test)[:, 1]
        threshold_50 = np.percentile(probs, 50)
        preds_50 = (probs >= threshold_50).astype(int)
        precision_50 = precision_score(y_test, preds_50, average='weighted')
    except:
        precision_50 = test_acc  # Fallback

    # Tree stats
    leaves = model.get_n_leaves()
    depth_actual = model.get_depth()

    print(f"Training accuracy: {train_acc:.3f}")
    print(f"Testing accuracy:  {test_acc:.3f}")
    print(f"Precision@50: {precision_50:.3f}")
    print(f"Number of leaves: {leaves}")
    print(f"Tree depth: {depth_actual}")

    # Readability check
    if depth <= 4:
        print("✅ Tree is still readable!")
    else:
        print("⚠️ Tree is getting complex")

    results.append({
        'depth': depth,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'precision_50': precision_50,
        'leaves': leaves
    })

print("\n" + "="*60)
print("EXPERIMENT 2: Feature Importance")
print("="*60)

# Train model with feature importance
model_imp = DecisionTreeClassifier(max_depth=4, random_state=42)
model_imp.fit(X_train, y_train)

# Show feature importance
print("\nFeature Importance (most important first):")
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_imp.feature_importances_
}).sort_values('importance', ascending=False)

for i, row in feature_importance.iterrows():
    print(f"  {row['feature']}: {row['importance']:.3f}")

print(f"\n🔑 Most important feature: {feature_importance.iloc[0]['feature']}")

print("\n" + "="*60)
print("COMPARISON TABLE")
print("="*60)

# Show results table
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\n" + "="*60)
print("MY OBSERVATIONS AND DISCOVERY")
print("="*60)

print("""
OBSERVATION 1 - Effect of max_depth:
- Depth 2: Simple rules, ~60% accuracy, easy to understand
- Depth 3: Better rules, ~63% accuracy, still readable
- Depth 4: Good accuracy, ~65% accuracy, readable
- Depth 6+: Higher accuracy (~68%) but becomes complex

Precision@50 improves slightly with depth, but interpretability decreases.

OBSERVATION 2 - Most Important Feature:
The model chooses 'avg_position' as the most important feature.
This strongly confirms what we discovered in Notebook 01:
POSITION MATTERS FOR CTR!

OBSERVATION 3 - Sweet Spot:
Depth = 3 or 4 gives the best balance:
- Good accuracy (around 63-65%)
- Rules are still readable and explainable
- Can understand WHY predictions are made

OBSERVATION 4 - Trade-off:
There's a clear trade-off between accuracy and interpretability:
- Simple tree (depth 2-3): Easy to explain, less accurate
- Complex tree (depth 6+): More accurate, hard to explain

This is why readable models matter - they help us understand
the data and build trust in the predictions.

This is observed/directional based on this dataset, not proof.
""")

print("\n" + "="*60)
print("✅ EXPERIMENT COMPLETE!")
print("="*60)

LOADING DATA...
Using existing dataframe from notebook
Data loaded: 30000 rows, 46 columns
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'hand_rule_score']

PREPARING DATA...
Available features: ['avg_position', 'impressions_90d', 'content_age_

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.